In [1]:
! pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content = "Hello World!",
    metadata = {"source" : "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Step 2: Import required libraries
import os
import shutil

# Step 3: Define the Google Drive folder
DRIVE_DIR = "/content/drive/MyDrive/Dataset_ap"

# Step 4: Create the folder structure expected by your original notebook
os.makedirs("data/pdfs", exist_ok=True)
os.makedirs("data/vector_store", exist_ok=True)

# Step 5: Copy Python.txt into data/
shutil.copy2(
    os.path.join(DRIVE_DIR, "Python.txt"),
    "data/Python.txt"
)

# Step 6: Copy both PDFs into data/pdfs/
shutil.copy2(
    os.path.join(DRIVE_DIR, "research1.pdf"),
    "data/pdfs/research1.pdf"
)

shutil.copy2(
    os.path.join(DRIVE_DIR, "research2.pdf"),
    "data/pdfs/research2.pdf"
)

# Step 7: Verify the files
print("Python.txt exists:", os.path.exists("data/Python.txt"))
print("research1.pdf exists:", os.path.exists("data/pdfs/research1.pdf"))
print("research2.pdf exists:", os.path.exists("data/pdfs/research2.pdf"))

print("\nFiles are ready. You can run your original notebook cells.")

Python.txt exists: True
research1.pdf exists: True
research2.pdf exists: True

Files are ready. You can run your original notebook cells.


In [8]:
# text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt" , encoding="utf-8")

/tmp/ipykernel_11721/1363355897.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [9]:
document = loader.load()

In [10]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [11]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader     # for simple pdf

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [12]:
# PDF data
from langchain_community.document_loaders.pdf import PyMuPDFLoader    # for complex pdfs
pdf_loader = PyMuPDFLoader(
    "/content/drive/MyDrive/Dataset_ap/research2.pdf"
)

document = pdf_loader.load()

print("Total pages:", len(document))
print(document[0].page_content[:500])

Total pages: 21
1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiongb, Xinyu Gaob, Kangxiang Jiab, Jinliu Panb, Yuxi Bic, Yi Daia, Jiawei Suna, Meng
Wangc, and Haofen Wang a,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University
cCollege of Design and Innovation, Tongji University
Abstract—Large Language Models (LLMs) showcase impres-
sive capabilities but e


**Ingestion Pipeline**

In [13]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

**Documents**

In [14]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [15]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [16]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

**Chunks**

In [17]:
# chunks
!pip install langchain_text_splitters

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_docs(documents , chunk_size=500, chunk_overlap=50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )
  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [19]:
chunks = split_docs(all_pdf_documents)


In [20]:
len(chunks)

244

**Embedding**

In [21]:
from sentence_transformers import SentenceTransformer

In [22]:
class EmbeddingManager:
  def __init__(self,model_name = "all-MiniLM-L6-v2"):
    self.model_name = model_name
    print("loading model ...",self.model_name)
    self.model = SentenceTransformer(self.model_name)
    print("embedding dimensions" , self.model.get_sentence_embedding_dimension())

  def generate_embeddings(self,text):
    embeddings = self.model.encode(text , show_progress_bar = True)
    print("embeddings shape:" , embeddings.shape)
    return embeddings

In [23]:
embedding_manager = EmbeddingManager()

loading model ... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions 384


/tmp/ipykernel_11721/1523572330.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions" , self.model.get_sentence_embedding_dimension())


**Vector Store**

In [24]:
import chromadb
import uuid

In [38]:
vector_store.client.delete_collection(
    name=vector_store.collection_name
)

vector_store._initialize_store()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [44]:


class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        # Create client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        # Create or retrieve collection with cosine similarity
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for pdf embeddings in RAG",
                "hnsw:space": "cosine"
            }
        )
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # Add batch to collection
        if documents_content:
            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store =", len(documents_content))
        print("docs in collection:", self.collection.count())

In [46]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [49]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)   # 320 chunks were there and for each chunk 384 dimension has come

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
total documents added in vector store = 244
docs in collection: 244


**Retrieval Pipeline**

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

In [62]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=-1.0):
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("no documents found")

        return retrieved_docs

In [63]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [64]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_e285bbfa-9b10-4f20-a1a5-5480913fb2d8',
  'document': 'integrity and compression ratio. PRCA tackled this issue by\ntraining an information extractor [69]. Similarly, RECOMP\nadopts a comparable approach by training an information\ncondenser using contrastive learning [71]. Each training data\npoint consists of one positive sample and five negative sam-\nples, and the encoder undergoes training using contrastive loss\nthroughout this process [102] .\nIn addition to compressing the context, reducing the num-',
  'metadata': {'producer': 'pdfTeX-1.40.25',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'creator': 'LaTeX with hyperref',
   'doc_index': 101,
   'keywords': '',
   'title': '',
   'trapped': '/False',
   'subject': '',
   'page': 9,
   'source': 'data/pdfs/research2.pdf',
   'total_pages': 21,
   'content_length': 450,
   'author': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 20

#Integrate with LLM

# OpenAI -GPT

In [75]:
from google.colab import userdata

API_KEY_OPENAI = userdata.get("OPENAI_API_KEY")

In [74]:
!pip install langchain-openai

In [79]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-6-luna",
    max_tokens=1024
)

In [80]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [81]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [82]:
print(answer)

An **encoder–decoder** is a model architecture with two parts:

- The **encoder** converts an input—such as text, an image, or audio—into a useful internal representation.
- The **decoder** uses that representation to generate an output, such as a translation, caption, or response.

For example, an image encoder can represent an image in a form a language model can use, and the decoder can generate a text description of it. In the context provided, encoders also turn text chunks into vector representations for retrieval.


#**Groq**

In [83]:
from google.colab import userdata

API_KEY_GROQ = userdata.get("GROQ_API_Key")

In [66]:
!pip install langchain-groq

In [84]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="qwen/qwen3.8-27b",
    temperature=0.1,
    max_tokens=1024
)

In [85]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [86]:
answer = generate_output("What is encoder decoder", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [87]:
print(answer)

Based on the provided context, an **encoder-decoder** is a type of language model architecture that is capable of performing **in-context learning with retrieval augmentation**.

This is explicitly mentioned in the reference to the "Raven" model, which is described as a "retrieval augmented encoder-decoder language model."


#**Gemini**

In [88]:
from google.colab import userdata

API_KEY_GEMINI = userdata.get("GEMINI_API_Key")

In [89]:
%pip install -q -U langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [106]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=API_KEY_GEMINI,
    max_output_tokens=1024
)

In [107]:
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n".join(
        [doc["document"] for doc in results]
    ) if results else ""

    if not context:
        return "We found no relevant context for the given query."

    prompt = f"""
    Answer the question using the provided context.

    If the context does not contain the answer, say so.
    Do not invent facts.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    # Handle both string and list-based responses
    if isinstance(response.content, list):
        answer = "\n".join(
            block["text"]
            for block in response.content
            if isinstance(block, dict) and block.get("type") == "text"
        )
    else:
        answer = response.content

    return answer

In [108]:
answer = generate_output("What is encoder decoder", rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [109]:
print(answer)

The provided context does not contain information to answer what an encoder-decoder is.
